In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class CrossAttention(nn.Module):
    def __init__(self, in_channels, context_dim, num_heads=12, attn_p=0, proj_p=0):
        super().__init__()

        self.num_heads = num_heads
        # in_channels muss ohne Rest durch num_heads teilbar sein
        self.head_dim = in_channels // num_heads 
        self.scale = self.head_dim ** -0.5

        # Query: Was suche ich? -> Kommt von den Punkten (in_channels)
        self.query = nn.Linear(in_channels, in_channels) 
        
        # Key & Value: Was habe ich zu bieten? -> Kommt vom Bild (context_dim)
        # Wir projizieren die Bild-Features so, dass sie am Ende die gleiche
        # Dimension (in_channels) haben wie unsere Punkte.
        self.key = nn.Linear(context_dim, in_channels)
        self.value = nn.Linear(context_dim, in_channels)

        self.attn_p = attn_p
        self.proj = nn.Linear(in_channels, in_channels)
        self.proj_drop = nn.Dropout(proj_p)
    
    def forward(self, x, context):
        # x: Deine Punkte (Batch, Anzahl_Punkte, in_channels)
        batch_size, seq_len, embed_dim = x.shape
        
        # context: Deine Bild-Features (Batch, Anzahl_Bild_Patches, context_dim)
        _, context_len, _ = context.shape 

        # Q wird aus 'x' (den Punkten) berechnet
        q = self.query(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # K und V werden aus 'context' (dem Bild) berechnet
        k = self.key(context).reshape(batch_size, context_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.value(context).reshape(batch_size, context_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Die PyTorch Attention-Funktion stört sich nicht daran, dass Q eine andere 
        # Sequenzlänge (Anzahl_Punkte) hat als K und V (Anzahl_Bild_Patches). 
        # Sie verbindet einfach jeden Punkt mit jedem Bild-Patch.
        x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_p)
        
        # Das Ergebnis hat automatisch wieder die Länge deiner Punkte (seq_len)!
        x = x.transpose(1, 2).reshape(batch_size, seq_len, embed_dim)
        x = self.proj_drop(self.proj(x))
        return x

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_channels, num_heads=12, attn_p=0, proj_p=0):
        super().__init__()

        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.scale = self.head_dim ** -0.5

        self.query = nn.Linear(in_channels, in_channels)        # Diese Daten werden nach und nach angepasst, starten random und werden dann durch das Training immer besser. 
        self.value = nn.Linear(in_channels, in_channels)        # Es ist wichtig, dass die Dimensionen der Query, Key und Value gleich sind, damit wir die Attention berechnen können.
        self.key = nn.Linear(in_channels, in_channels)          

        self.attn_p = attn_p
        self.proj_p = nn.Linear(in_channels, in_channels)
        self.proj_drop = nn.Dropout(proj_p)
    
    def forward(self, x):
        batch_size, seq_len, embed_dim = x.shape

        q = self.query(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).reshape(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_p)
        
        x = x.transpose(1,2).reshape(batch_size, seq_len, embed_dim)
        x = self.proj_drop(self.proj_p(x))
        return x


In [5]:
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) Block.
    
    Warum ist dieses MLP so wichtig?
    Während die Self-Attention-Schicht dafür zuständig ist, Informationen 
    ZWISCHEN verschiedenen Pixeln oder Sequenzpositionen auszutauschen 
    (räumlicher Kontext), arbeitet das MLP isoliert auf jedem einzelnen Pixel. 
    
    Es nimmt die durch die Attention neu gesammelten Informationen (die Features/Kanäle) 
    eines Pixels und verarbeitet diese tiefgreifend in sich selbst. Das MLP mischt 
    also nur entlang der Feature-Dimension. 
    
    Durch die Expansion der Kanäle (meist das 2- bis 4-fache in der Mitte) und 
    die nicht-lineare Aktivierungsfunktion (GELU) erhält das Modell hier den 
    nötigen "Denkraum", um komplexe Muster zu speichern und die gesammelten 
    Erkenntnisse der Attention zu festigen.
    
    Zusammenfassung der Aufgabenteilung im Transformer:
    - Attention = Informationsaustausch über das gesamte Bild (Kommunikation).
    - MLP = Tiefe Verarbeitung der gesammelten Informationen pro Pixel (Einzelarbeit).
    """
    def __init__(self, in_channels, mlp_ratio=4, mlp_p=0):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, in_channels * mlp_ratio)
        self.act = nn.GELU()
        self.drop1 = nn.Dropout(mlp_p)
        self.fc2 = nn.Linear(in_channels * mlp_ratio, in_channels)
        self.drop2 = nn.Dropout(mlp_p)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop1(x)

        x= self.fc2(x)
        x = self.drop2(x)
        return x
    


In [6]:
class PointTransformerBlock(nn.Module):
    def __init__(self,
                 in_channels,
                 context_dim,       
                 num_heads=4, 
                 mlp_ratio=2,
                 proj_p=0,
                 attn_p=0,
                 mlp_p=0):
        
        super().__init__()
        
        # 1. Self-Attention (Punkte reden mit Punkten)
        self.norm1 = nn.LayerNorm(in_channels, eps=1e-6)
        self.attn = SelfAttention(in_channels=in_channels,
                                  num_heads=num_heads, 
                                  attn_p=attn_p,
                                  proj_p=proj_p)
        
        # 2. Cross-Attention (Punkte schauen aufs Bild)
        self.norm2 = nn.LayerNorm(in_channels, eps=1e-6)
        self.cross_attn = CrossAttention(in_channels=in_channels, 
                                         context_dim=context_dim, 
                                         num_heads=num_heads)
        
        # 3. MLP (Punkte verarbeiten das Gesehene isoliert)
        self.norm3 = nn.LayerNorm(in_channels, eps=1e-6)
        self.mlp = MLP(in_channels=in_channels,
                       mlp_ratio=mlp_ratio,
                       mlp_p=mlp_p)
        
    def forward(self, x, image_features):
        # x kommt hier schon als (Batch, n_punkte, in_channels) an.
        # Kein Reshape mehr nötig! Die Punkte können direkt durchfließen.
        
        x = x + self.attn(self.norm1(x))
        x = x + self.cross_attn(self.norm2(x), context=image_features)
        x = x + self.mlp(self.norm3(x))

        return x

In [ ]:
import torch
from torch import nn

class ContourDiffusionModel(nn.Module):
    def __init__(self, n_punkte, in_channels, context_dim, num_blocks=6):
        super().__init__()
        
        self.n_punkte = n_punkte
        
        # 1. HIER IST DEIN POSITIONAL ENCODER!
        # Er lernt für jede der 'n_punkte' Positionen einen eigenen Vektor der Größe 'in_channels'
        self.pos_embed = nn.Embedding(n_punkte, in_channels)
        
        # 2. Deine Transformer-Blöcke
        # Wir erstellen eine Liste mit z.B. 6 hintereinandergeschalteten Blöcken
        self.blocks = nn.ModuleList([
            PointTransformerBlock(
                in_channels=in_channels, 
                context_dim=context_dim
            ) for _ in range(num_blocks)
        ])
        
        # Optional: Hier würden noch lineare Schichten stehen, die ganz am Anfang 
        # aus (x, y) z.B. 256 Kanäle machen, und ganz am Ende aus 256 wieder (x, y).
        # self.linear_in = nn.Linear(2, in_channels)
        # self.linear_out = nn.Linear(in_channels, 2)

    def forward(self, x, image_features):
        # x ist dein verrauschtes Signal, z.B. schon aufgebläht auf 'in_channels'
        # Form: (Batch, n_punkte, in_channels)
        
        # 1. Positionen generieren
        # Erstellt eine Liste von [0, 1, 2, ..., n_punkte-1]
        # 'device=x.device' stellt sicher, dass die Liste auf der gleichen Grafikkarte liegt wie 'x'
        pos_ids = torch.arange(self.n_punkte, device=x.device)
        
        # 2. Positionen als Vektoren abrufen und auf die Punkte addieren
        positionen = self.pos_embed(pos_ids)
        x = x + positionen 
        # JETZT wissen die Punkte, an welcher Stelle der Kontur sie sich befinden!
        
        # 3. Durch alle Transformer-Blöcke schleusen
        for block in self.blocks:
            x = block(x, image_features)
            
        return x
    
    

In [8]:
"""
DEINE PUNKTE (x)                      DEIN BILD (image_features)
           |                                          |
           v                                          v
   +---------------+                  +-------------------------------+
   | nn.Linear (Q) |                  | nn.Linear (K) | nn.Linear (V) |
   +---------------+                  +-------------------------------+
           |                                  |               |
           v                                  v               |
        [ Query ]                          [ Key ]         [ Value ]
   (Was suche ich?)                  (Was biete ich?)   (Der Inhalt)
           |                                  |               |
           |        1. MATCHING (Q × K^T)     |               |
           +----------------------------------+               |
                            |                                 |
                            v                                 |
                 [ Attention-Matrix ]                         |
      (Matrix: Welcher Punkt schaut auf welches Pixel?)       |
                            |                                 |
                            |         2. ABHOLEN (A × V)      |
                            +---------------------------------+
                                              |
                                              v
                                          [ Output ]
                            (Punkte angereichert mit Bild-Wissen)""";